# Example 0.9.1 / 1: Solve an Instance

Companion notebook for **0.9.1 The Diet Model** in the AMPL lab Documentation, section "Solve an Instance". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve an Instance

The AMPL book's first diet problem has eight prepared dinners and four vitamins. A serving supplies a percentage of a day's requirement, and a week needs 700% of each. In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read `model.mod`; the nutrient table is turned into a dictionary keyed by `(nutrient, food)` pairs, and `to_pandas()` returns the purchases:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set FOOD;                         # Available foods
set NUTR;                         # Required nutrients
param cost {FOOD} >= 0;           # Dollars per serving
param amt {NUTR, FOOD} >= 0;      # Nutrient units per serving
param need {NUTR} >= 0;           # Required nutrient units
var Buy {FOOD} >= 0;              # Servings purchased
minimize TotalCost: sum {f in FOOD} cost[f] * Buy[f];
subject to NutrientNeed {n in NUTR}:
    sum {f in FOOD} amt[n,f] * Buy[f] >= need[n];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
nutr = ["A", "C", "B1", "B2"]
pct = {  # percent of a day's requirement per serving, in the order of nutr
    "BEEF": [60, 20, 10, 15], "CHK": [8, 0, 20, 20], "FISH": [8, 10, 15, 10],
    "HAM": [40, 40, 35, 10], "MCH": [15, 35, 15, 15], "MTL": [70, 30, 15, 15],
    "SPG": [25, 50, 25, 15], "TUR": [60, 20, 15, 10],
}
ampl.set["NUTR"] = nutr
ampl.set["FOOD"] = list(pct)
ampl.param["cost"] = {"BEEF": 3.19, "CHK": 2.59, "FISH": 2.29, "HAM": 2.89,
                      "MCH": 1.89, "MTL": 1.99, "SPG": 1.99, "TUR": 2.49}
ampl.param["amt"] = {(n, f): pct[f][k] for f in pct for k, n in enumerate(nutr)}
ampl.param["need"] = {n: 700 for n in nutr}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
buy = ampl.var["Buy"].to_pandas()
print(buy[buy["Buy.val"] > 1e-6].round(4))   # only the foods bought

HiGHS 1.14.0: optimal solution; objective 88.2
2 simplex iterations
0 barrier iterations
solved
     Buy.val
MCH  46.6667


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set NUTR := A C B1 B2;
set FOOD := BEEF CHK FISH HAM MCH MTL SPG TUR;
param cost := BEEF 3.19  CHK 2.59  FISH 2.29  HAM 2.89
              MCH 1.89   MTL 1.99  SPG 1.99   TUR 2.49;
param amt:  BEEF  CHK  FISH  HAM  MCH  MTL  SPG  TUR :=
  A          60     8     8   40   15   70   25   60
  C          20     0    10   40   35   30   50   20
  B1         10    20    15   35   15   15   25   15
  B2         15    20    10   10   15   15   15   10 ;
param need := A 700  C 700  B1 700  B2 700;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display solve_result, TotalCost;
display {f in FOOD: Buy[f] > 1e-6} Buy[f];

HiGHS 1.14.0: optimal solution; objective 88.2
2 simplex iterations
0 barrier iterations
solve_result = solved
TotalCost = 88.2

Buy[f] [*] :=
MCH  46.6667
;

